In [3]:
import pandas as pd
import csv

In [9]:
def clean_mario_kart_data(input_file, output_file):
    """
    清洗马里奥卡丁车世界纪录数据集，只保留玩家信息、分数、角色、车辆、滑翔伞和轮胎的信息。
    
    参数:
        input_file (str): 输入CSV文件路径
        output_file (str): 输出CSV文件路径
    """
    # 由于原始CSV可能存在解析问题，我们使用更灵活的方式读取数据
    data = []
    
    with open(input_file, 'r', encoding='utf-8') as file:
        lines = file.readlines()
        
        for line in lines:
            # 处理CSV行，考虑到引号内的逗号
            row = []
            in_quotes = False
            current_field = ""
            
            for char in line:
                if char == '"':
                    in_quotes = not in_quotes
                elif char == ',' and not in_quotes:
                    row.append(current_field.strip('"'))
                    current_field = ""
                else:
                    current_field += char
                    
            # 添加最后一个字段
            if current_field:
                row.append(current_field.strip('"'))
                
            data.append(row)
    
    # 设置列名（基于数据分析）
    column_names = [
        "Record ID", "Player Name", "Record Date", "Track Name", "Score", 
        "Track ID", "Country", "Video URL", "Combo parameters1", "Combo parameters2", 
        "Vehicles", "Tires", "Unknown1", "Gliders", "Unknown2", 
        "Drivers", "Lap1 Time", "Lap2 Time", "Lap3 Time", "Unknown3", 
        "Unknown4", "Extra1", "Extra2", "Extra3", "Extra4", "CC"
    ]
    
    # 创建DataFrame
    df = pd.DataFrame(data, columns=column_names)
    
    # 保留需要的列
    columns_to_keep = [
        "Player Name",  # 玩家信息
        "Score",        # 分数
        "Drivers",    # 角色
        "Vehicles",      # 车辆
        "Gliders",       # 滑翔伞
        "Tires"         # 轮胎
    ]
    
    df_cleaned = df[columns_to_keep]
    
    # 清洗数据：移除引号，处理缺失值
    for col in df_cleaned.columns:
        # 移除可能的多余引号
        df_cleaned[col] = df_cleaned[col].str.strip('"')
        
        # 将"-"标记为空值
        df_cleaned[col] = df_cleaned[col].replace("-", pd.NA)
    
    # 保存清洗后的数据
    df_cleaned.to_csv(output_file, index=False, encoding='utf-8')
    
    print(f"数据清洗完成! 保存到 {output_file}")
    print(f"保留的列: {', '.join(columns_to_keep)}")
    print(f"总行数: {len(df_cleaned)}")
    
    # 返回基本统计信息
    return {
        "total_rows": len(df_cleaned),
        "columns": columns_to_keep,
        "missing_values": df_cleaned.isna().sum().to_dict(),
        "sample": df_cleaned.head(5).to_dict('records')
    }

# 使用示例
if __name__ == "__main__":
    input_file = "wr.csv"  # 输入文件路径
    output_file = "mario_kart_records_cleaned.csv"  # 输出文件路径
    
    # 执行数据清洗
    stats = clean_mario_kart_data(input_file, output_file)
    
    # 打印数据样例
    print("\n数据样例:")
    for i, row in enumerate(stats["sample"]):
        print(f"样例 {i+1}:")
        for col, val in row.items():
            print(f"  {col}: {val}")
        print()

数据清洗完成! 保存到 mario_kart_records_cleaned.csv
保留的列: Player Name, Score, Drivers, Vehicles, Gliders, Tires
总行数: 13103

数据样例:
样例 1:
  Player Name: B!KZO
  Score: 79969
  Drivers: Morton
  Vehicles: Streetle
  Gliders: Plane Glider
  Tires: Slim

样例 2:
  Player Name: Mattew
  Score: 114981
  Drivers: Heavy Mii
  Vehicles: Blue Falcon
  Gliders: Super Glider
  Tires: Slim

样例 3:
  Player Name: Mattew
  Score: 123606
  Drivers: None
  Vehicles: None
  Gliders: None
  Tires: None

样例 4:
  Player Name: Del
  Score: 98522
  Drivers: Male Villager
  Vehicles: Standard Kart
  Gliders: Super Glider
  Tires: Off-Road

样例 5:
  Player Name: DestinyCst
  Score: 97993
  Drivers: None
  Vehicles: None
  Gliders: None
  Tires: None



/var/folders/r6/0pr4ddf51zzb1wwg7_yt8vp40000gn/T/ipykernel_7241/2353262982.py:63: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_cleaned[col] = df_cleaned[col].str.strip('"')
/var/folders/r6/0pr4ddf51zzb1wwg7_yt8vp40000gn/T/ipykernel_7241/2353262982.py:66: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_cleaned[col] = df_cleaned[col].replace("-", pd.NA)


In [11]:
import pandas as pd
import csv

def clean_mario_kart_data(input_file, output_file):
    """
    清洗马里奥卡丁车世界纪录数据集，只保留玩家信息、分数、角色、车辆、滑翔伞和轮胎的信息。
    并删除所有 Character, Vehicle, Glider, Tires 为 None 的数据行。
    
    参数:
        input_file (str): 输入CSV文件路径
        output_file (str): 输出CSV文件路径
    """
    print(f"开始处理文件: {input_file}")
    
    # 直接使用pandas读取CSV文件，尝试自动处理引号和分隔符
    try:
        # 首先尝试使用默认设置读取
        df = pd.read_csv(input_file, encoding='utf-8')
    except Exception as e:
        print(f"标准读取失败，尝试自定义读取方式: {e}")
        
        # 如果默认读取失败，使用更多选项尝试读取
        df = pd.read_csv(input_file, encoding='utf-8', quotechar='"', 
                         escapechar='\\', on_bad_lines='warn', 
                         low_memory=False)
    
    # 打印列名，以便诊断
    print("原始数据的列名:")
    print(df.columns.tolist())
    
    # 查看数据的形状
    print(f"原始数据形状: {df.shape}")
    
    # 根据我们的分析，我们可以推断出以下列映射
    # 注意：这里使用实际存在于DataFrame中的列名
    # 1. 找到玩家名称的列：通常是第2列
    player_col = df.columns[1] if len(df.columns) > 1 else None
    
    # 2. 找到分数的列：通常是第5列
    score_col = df.columns[4] if len(df.columns) > 4 else None
    
    # 3. 找到角色的列：通常是第16列
    character_col = df.columns[15] if len(df.columns) > 15 else None
    
    # 4. 找到车辆的列：通常是第11列
    vehicle_col = df.columns[10] if len(df.columns) > 10 else None
    
    # 5. 找到滑翔伞的列：通常是第14列
    glider_col = df.columns[13] if len(df.columns) > 13 else None
    
    # 6. 找到轮胎的列：通常是第12列
    tires_col = df.columns[11] if len(df.columns) > 11 else None
    
    # 创建列名映射字典
    column_mapping = {
        player_col: "Player Name",
        score_col: "Score",
        character_col: "Character",
        vehicle_col: "Vehicle",
        glider_col: "Glider",
        tires_col: "Tires"
    }
    
    # 筛选并重命名列
    columns_to_keep = list(column_mapping.keys())
    
    # 检查并移除None值
    columns_to_keep = [col for col in columns_to_keep if col is not None]
    
    print("将保留的列:")
    for i, col in enumerate(columns_to_keep):
        print(f"{i+1}. {col} -> {column_mapping.get(col, 'N/A')}")
    
    # 检查是否找到了所有需要的列
    if not columns_to_keep:
        raise ValueError("无法找到需要保留的列!")
    
    # 只保留需要的列
    df_cleaned = df[columns_to_keep].copy()
    
    # 重命名列
    # 仅重命名存在的列
    rename_mapping = {k: v for k, v in column_mapping.items() if k in df_cleaned.columns}
    df_cleaned.rename(columns=rename_mapping, inplace=True)
    
    # 清洗数据：处理缺失值和格式
    for col in df_cleaned.columns:
        # 将列转换为字符串类型
        df_cleaned[col] = df_cleaned[col].astype(str)
        
        # 移除可能的多余引号
        df_cleaned[col] = df_cleaned[col].str.strip('"')
        
        # 将"-"标记为空值
        df_cleaned[col] = df_cleaned[col].replace("-", None)
        
        # 将空字符串标记为空值
        df_cleaned[col] = df_cleaned[col].replace("", None)
    
    # 统计过滤前的行数
    print(f"过滤前的总行数: {len(df_cleaned)}")
    
    # 删除Character, Vehicle, Glider, Tires为None的行
    important_cols = ["Character", "Vehicle", "Glider", "Tires"]
    important_cols = [col for col in important_cols if col in df_cleaned.columns]
    
    # 在过滤前输出各列的空值数量
    print("各列的空值数量:")
    for col in important_cols:
        null_count = df_cleaned[col].isna().sum()
        print(f"{col}: {null_count} 行为空")
    
    # 过滤掉任何一个重要列为空的行
    df_filtered = df_cleaned.dropna(subset=important_cols)
    
    # 统计过滤后的行数
    print(f"过滤后的总行数: {len(df_filtered)}")
    print(f"删除了 {len(df_cleaned) - len(df_filtered)} 行数据")
    
    # 保存清洗后的数据
    df_filtered.to_csv(output_file, index=False, encoding='utf-8')
    
    print(f"数据清洗完成! 保存到 {output_file}")
    print(f"保留的列: {', '.join(df_filtered.columns)}")
    
    # 返回基本统计信息
    return {
        "total_rows_before_filtering": len(df_cleaned),
        "total_rows_after_filtering": len(df_filtered),
        "rows_removed": len(df_cleaned) - len(df_filtered),
        "columns": df_filtered.columns.tolist(),
        "missing_values": df_filtered.isna().sum().to_dict(),
        "sample": df_filtered.head(5).to_dict('records')
    }

# 使用示例
if __name__ == "__main__":
    input_file = "wr.csv"  # 输入文件路径
    output_file = "mario_kart_records_cleaned.csv"  # 输出文件路径
    
    try:
        # 执行数据清洗
        stats = clean_mario_kart_data(input_file, output_file)
        
        # 打印数据样例
        print("\n数据样例:")
        for i, row in enumerate(stats["sample"]):
            print(f"样例 {i+1}:")
            for col, val in row.items():
                print(f"  {col}: {val}")
            print()
    except Exception as e:
        print(f"处理过程中出现错误: {e}")
        import traceback
        traceback.print_exc()

开始处理文件: wr.csv
原始数据的列名:
['2588', 'B!KZO', '2017-04-28', 'SNES Donut Plains 3', '79969', '22', 'Japan', 'N/A', '7-0-0', '1-1-1', 'Streetle', 'Slim', '0', 'Plane Glider', '0.1', 'Morton', '27.378', '26.351', '26.240', '1', '0.2', 'Unnamed: 21', 'Unnamed: 22', 'Unnamed: 23', 'Unnamed: 24', '150']
原始数据形状: (13102, 26)
将保留的列:
1. B!KZO -> Player Name
2. 79969 -> Score
3. Morton -> Character
4. Streetle -> Vehicle
5. Plane Glider -> Glider
6. Slim -> Tires
过滤前的总行数: 13102
各列的空值数量:
Character: 231 行为空
Vehicle: 230 行为空
Glider: 243 行为空
Tires: 231 行为空
过滤后的总行数: 12856
删除了 246 行数据
数据清洗完成! 保存到 mario_kart_records_cleaned.csv
保留的列: Player Name, Score, Character, Vehicle, Glider, Tires

数据样例:
样例 1:
  Player Name: Mattew
  Score: 114981
  Character: Heavy Mii
  Vehicle: Blue Falcon
  Glider: Super Glider
  Tires: Slim

样例 2:
  Player Name: Del
  Score: 98522
  Character: Male Villager
  Vehicle: Standard Kart
  Glider: Super Glider
  Tires: Off-Road

样例 3:
  Player Name: Alexal
  Score: 49092
  Character: K

In [15]:
wr = pd.read_csv("mario_kart_records_cleaned.csv")
wr.head()

,Player Name,Score,Character,Vehicle,Glider,Tires
0,Mattew,114981,Heavy Mii,Blue Falcon,Super Glider,Slim
1,Del,98522,Male Villager,Standard Kart,Super Glider,Off-Road
2,Alexal,49092,King Boo,W 25 Silver Arrow,Super Glider,Button
3,Danny,110439,Morton,Splat Buggy,Super Glider,Slim
4,Mattew,93528,King Boo,Blue Falcon,Super Glider,Slim
